# Deploying LLMs: vLLM and SGLang in Practice

> Notebook 25 ended with a deployment-tool selection table: llama.cpp for local CPU, vLLM and SGLang for GPU servers, TensorRT-LLM for maximum performance. But there is still a gap between "picking a tool" and "actually getting the model to serve traffic."
>
> This section deploys a real language model with vLLM and SGLang. The first half walks through the full pipeline with the off-the-shelf Qwen2.5-0.6B: launch the server, send requests, stream output, and batch prompts. The second half tackles the hardest case -- when the model you trained yourself has a brand-new architecture and vocabulary, how do you make vLLM and SGLang recognize it?

Deployment is fundamentally about turning a trained checkpoint into a service that can be called many times. It sounds straightforward, but as soon as concurrency, memory, and latency enter the picture, a cascade of engineering problems surfaces: how do simultaneous requests queue? What happens when KV Cache fills up GPU memory? How do we batch long and short requests together?

vLLM and SGLang are the two most widely used open-source inference frameworks today (docs: [vLLM docs](https://docs.vllm.ai/), [SGLang docs](https://docs.sglang.ai/)). Both are written in Python on top of PyTorch, but they take different optimization paths. vLLM introduced PagedAttention, treating KV Cache like virtual memory pages; SGLang introduced RadixAttention, caching shared prefixes across requests. Both share a common foundation -- continuous batching -- which lets new requests join or leave an in-flight batch at any time.

This section first builds intuition for these optimizations, then runs through the full pipeline with Qwen2.5-0.6B, and finally handles the registration problems for custom architectures and custom vocabularies. Some cells involve launching an inference server and require a GPU to actually run; these cells are marked at the top, and readers without a GPU can still read the code to understand the API.

## 1. Why not just use HuggingFace transformers directly

The most naive deployment is to load a model with HuggingFace transformers and write a Python loop that handles requests. This works, but throughput in production is very low.

The root cause is that transformers batching is too coarse: all requests must be packed into one batch before they can be computed together, and the longest sequence in the batch determines the compute time for every sequence. If a batch contains one 2000-token request and nine 50-token requests, the short ones must wait for the long one to finish before returning. New requests can only join after the current batch completes.

Both vLLM and SGLang solve this with continuous batching: a new request does not wait for the current batch to finish; it can join or leave the batch at every decode step. This keeps the GPU saturated and lets short requests return as soon as they are done.

In [ ]:
# Simulate the latency difference between static batching and continuous batching
# This cell does not need a GPU; pure algorithm demo

import random
random.seed(42)

# Static batching: pack a full batch, longest sequence determines time
def simulate_static_batching(requests, batch_size=4):
    total_time = 0
    for i in range(0, len(requests), batch_size):
        batch = requests[i:i+batch_size]
        batch_time = max(batch)  # entire batch time is set by the longest request
        total_time += batch_time
    return total_time

# Continuous batching: every decode step is a scheduling opportunity
# Finished requests leave immediately, the rest keep decoding
def simulate_continuous_batching(requests):
    remaining = list(requests)
    total_steps = 0
    while remaining:
        # every step, each remaining request decodes one token
        remaining = [r - 1 for r in remaining if r - 1 > 0]
        total_steps += 1
    return total_steps

# Simulate 8 requests with very different lengths
requests = [5, 200, 8, 150, 6, 180, 10, 120]

static_time = simulate_static_batching(requests, batch_size=4)
continuous_time = simulate_continuous_batching(requests)

print(f"Number of requests: {len(requests)}, lengths are {requests}")
print(f"\nStatic batching (batch=4) total steps: {static_time}")
print(f"Continuous batching total steps: {continuous_time}")
print(f"\nKey observation: the larger the length variance, the bigger the continuous batching advantage")
print("Short requests do not wait for long ones; they leave the batch as soon as they finish, keeping the GPU saturated")


## 2. PagedAttention and RadixAttention: two approaches to KV Cache management

Continuous batching solves the "when can a new request join" problem, but there is a harder engineering problem underneath: KV Cache memory management.

KV Cache is the historical key/value tensors cached during inference to avoid recomputing Attention. Its defining property is that its size grows dynamically with sequence length -- we do not know how many tokens a request will eventually generate, so we cannot pre-allocate a fixed amount of memory. The simplest approach is to pre-allocate the maximum length per request, but this wastes a lot of space (fragmentation) and ultimately limits how many requests can be served concurrently.

vLLM's PagedAttention borrows from the operating system's virtual-memory paging: KV Cache is sliced into fixed-size blocks (typically 16 tokens each) and allocated on demand. A request's KV Cache does not have to be contiguous; it can be scattered across many blocks, indexed by a block table. This raises memory utilization from ~20% to over 80%. Original paper: [Efficient Memory Management for Large Language Model Serving with PagedAttention](https://arxiv.org/abs/2309.06180) (SOSP 2023).

SGLang's RadixAttention takes another route: it organizes all requests' KV Caches into a Radix Tree, so requests sharing a prefix share the same KV. For example, if 100 users all ask "Please introduce Python", the KV for "Please introduce Python" is computed once and reused by subsequent requests. In multi-turn dialogue, few-shot prompting, and agent workflows, shared prefixes are very common, and RadixAttention's speedup is significant. Original paper: [SGLang: Efficient Execution of Structured Language Model Programs](https://arxiv.org/abs/2312.07104). Continuous batching itself first appeared in [Orca](https://www.usenix.org/conference/osdi22/presentation/yu) (OSDI 2022); both vLLM and SGLang build on top of it.

| Optimization | Origin | Core idea | Best fit |
|:---|:---|:---|:---|
| Continuous Batching | Generic | dynamically insert/leave requests during decode | any continuous inference service |
| PagedAttention | vLLM | page-based KV Cache management, reduces fragmentation | high concurrency, variable-length requests |
| RadixAttention | SGLang | reuse KV Cache for shared prefixes | multi-turn dialogue, long system prompts, agent workflows |

It is worth noting that vLLM later added its own Prefix Cache feature (Automatic Prefix Caching), and SGLang also supports paged management -- the gap between the two frameworks is shrinking. Still, the rule of thumb holds: pure throughput favors vLLM, structured output and complex shared prefixes favor SGLang.

## 3. Prepare the model: Qwen2.5-0.6B

This section uses Qwen2.5-0.6B as the example model. At 0.6B parameters, the checkpoint is small (~1.2 GB) and runs on a GPU with 4 GB of VRAM, which makes it a good starting point for a deployment tutorial.

The minimum file set for deploying a model:

- `config.json`: model architecture configuration (layers, hidden size, vocab size, etc.)
- `model.safetensors`: weights
- `tokenizer.json` or `tokenizer_config.json`: tokenizer
- `generation_config.json` (optional): default generation parameters

These files are downloaded from HuggingFace Hub. The cell below uses `huggingface_hub` to pull the model locally.

In [ ]:
# This cell requires network access to download (~1.2 GB)
# If huggingface_hub is not installed: pip install huggingface_hub
# On restricted networks, set a mirror: export HF_ENDPOINT=https://hf-mirror.com

from huggingface_hub import snapshot_download
import json
import os

model_path = snapshot_download(
    repo_id="Qwen/Qwen2.5-0.6B",
    allow_patterns=["*.json", "*.safetensors", "*.txt"],
)
print(f"Model downloaded to: {model_path}\n")

print("Model directory contents:")
for f in sorted(os.listdir(model_path)):
    size = os.path.getsize(os.path.join(model_path, f))
    size_str = f"{size/1e6:.1f} MB" if size > 1e6 else f"{size/1e3:.1f} KB"
    print(f"  {f:40s} {size_str}")

# config.json is the key file for deployment; it decides how the inference framework loads the model
print("\nKey fields in config.json:")
with open(os.path.join(model_path, "config.json")) as fp:
    config = json.load(fp)
for key in ["architectures", "hidden_size", "num_hidden_layers",
            "num_attention_heads", "vocab_size", "torch_dtype", "max_position_embeddings"]:
    print(f"  {key:30s} {config.get(key)}")

The most important field in `config.json` is `architectures` -- it tells the inference framework which model class to use when loading weights. Qwen2.5 has `architectures = ["Qwen2ForCausalLM"]`, and both vLLM and SGLang ship a built-in implementation, so it works out of the box.

Meaning of the other fields:

- `hidden_size`: hidden dimension of each transformer layer (1024 for Qwen2.5-0.6B)
- `num_hidden_layers`: number of transformer layers (28)
- `num_attention_heads`: number of attention heads (16)
- `vocab_size`: vocabulary size (151936; includes Chinese, code, special tokens)
- `max_position_embeddings`: maximum sequence length at training time (32768)

The `architectures` field is the central obstacle in the "deploy your own model" section later: if the framework does not recognize this string, it has no way to know which code to use to load the weights.

## 4. vLLM offline inference: the LLM class

vLLM offers two usage modes: offline inference and serving. Offline inference uses the `LLM` class directly in Python -- good for batch processing, benchmarks, and non-concurrent scenarios. Serving uses `vllm serve` to launch an OpenAI-compatible HTTP service -- good for production.

We start with offline inference.

In [ ]:
# This cell requires a GPU (vLLM has a hard CUDA dependency)
# On CPU it will raise: ValueError: No supported GPU is found
# Without a GPU, read the code to understand the API; run it later when GPU is available

# pip install vllm

from vllm import LLM, SamplingParams

# Create the LLM instance; vLLM loads all weights into GPU memory up front
# and reserves a fraction of memory (gpu_memory_utilization) as the KV Cache pool
llm = LLM(
    model=model_path,           # model path; can also be a HuggingFace repo_id
    gpu_memory_utilization=0.5, # use 50% of VRAM (leave the other half for other processes)
    max_model_len=2048,         # maximum sequence length; longer inputs are truncated
    dtype="float16",            # weight dtype; float16 uses half the VRAM of float32
)

# Sampling params: same generation strategies as Notebook 17
sampling = SamplingParams(
    temperature=0.7,   # higher temperature = more random output
    top_p=0.9,         # nucleus sampling; only sample from tokens covering 90% cumulative probability
    max_tokens=100,    # generate at most 100 tokens
)

# Pass multiple prompts at once; vLLM does continuous batching internally
prompts = [
    "Explain gradient descent in one sentence.",
    "Write a Python function to reverse a string.",
    "What is the capital of France?",
]

outputs = llm.generate(prompts, sampling)

for prompt, output in zip(prompts, outputs):
    text = output.outputs[0].text
    print(f"Prompt: {prompt}")
    print(f"Output: {text}\n")

Key parameters of the `LLM` class:

- `gpu_memory_utilization`: fraction of total VRAM that vLLM claims. 0.5 means it takes half, leaving the rest for other processes. At 0.9 vLLM grabs as much VRAM as possible for the KV Cache pool, supporting more concurrent requests, but other processes get nothing
- `max_model_len`: the maximum sequence length (input + output) the model can handle. This directly affects the KV Cache pool size -- a larger value can hold more tokens, but each request's ceiling is also higher
- `dtype`: weight precision. float16 is the default; bfloat16 trades a bit of precision for a wider numeric range; AWQ/GPTQ quantized models use their own quantized dtype

`SamplingParams` controls decoding behavior, the same generation strategies covered in Notebook 17 -- temperature, top-k, top-p, etc.

## 5. Launch an OpenAI-compatible service with vLLM

Offline inference is good for scripts and batch processing, but production usually launches an HTTP service that frontends, backends, and other microservices can call via REST. vLLM ships a `vllm serve` command that starts a server fully compatible with the OpenAI Chat Completions API -- any code that uses the `openai` Python client or calls OpenAI via HTTP works as-is, just by changing `base_url`.

We launch the server from the command line, not inside the notebook, because the server holds the foreground.

Run this in a terminal:

```bash
# Basic launch command
vllm serve Qwen/Qwen2.5-0.6B \
  --port 8000 \
  --gpu-memory-utilization 0.5 \
  --max-model-len 2048
```

On success you will see logs like:

```
INFO:     Started server process [12345]
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000
```

The server is now listening on port 8000 and accepts OpenAI-format requests. Common parameters for `vllm serve`:

| Parameter | Effect |
|:---|:---|
| `--port` | server port |
| `--host` | bind address; 0.0.0.0 allows external access |
| `--gpu-memory-utilization` | VRAM utilization fraction |
| `--max-model-len` | maximum sequence length |
| `--tensor-parallel-size` | multi-GPU tensor parallelism (for large models) |
| `--quantization` | quantization method (awq, gptq, fp8) |
| `--enable-prefix-caching` | enable prefix caching (vLLM's equivalent of RadixAttention) |

In [ ]:
# This cell requires vllm serve running in a terminal, and a GPU
# Without the server running you get ConnectionError

# Use stdlib requests; no extra dependency needed
import requests
import json

VLLM_URL = "http://localhost:8000"

# vLLM is fully compatible with OpenAI's /v1/chat/completions endpoint
response = requests.post(
    f"{VLLM_URL}/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-0.6B",  # must match the name passed to vllm serve
        "messages": [
            {"role": "user", "content": "Explain what a transformer is in one sentence."}
        ],
        "temperature": 0.7,
        "max_tokens": 100,
    },
    timeout=60,
)

data = response.json()
print("Status code:", response.status_code)
print("Reply:", data["choices"][0]["message"]["content"])
print("\nKey metadata:")
print(f"  finish reason: {data['choices'][0]['finish_reason']}")
print(f"  prompt tokens: {data['usage']['prompt_tokens']}")
print(f"  completion tokens: {data['usage']['completion_tokens']}")

In [ ]:
# This cell also requires vllm serve running

# Streaming: stream=True makes the server return tokens one at a time
# The frontend gets a typewriter effect, much better UX than waiting for the full reply

response = requests.post(
    f"{VLLM_URL}/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-0.6B",
        "messages": [
            {"role": "user", "content": "Write a short nature poem."}
        ],
        "stream": True,   # key: enable streaming
        "max_tokens": 80,
    },
    stream=True,          # requests also needs stream=True
    timeout=60,
)

# Read the SSE (Server-Sent Events) stream line by line
print("Streaming output:\n")
for line in response.iter_lines():
    if not line:
        continue
    line = line.decode("utf-8")
    if not line.startswith("data: "):
        continue
    payload = line[6:]
    if payload == "[DONE]":
        break
    chunk = json.loads(payload)
    delta = chunk["choices"][0]["delta"].get("content", "")
    print(delta, end="", flush=True)  # no newline; print continuously

print("\n\n-> The server pushes tokens one at a time via the SSE protocol")


## 6. Deploy the same model with SGLang

SGLang's workflow is almost identical to vLLM -- launch a server, send OpenAI-compatible requests. The differences are mostly in the launch command and a few server-side optimizations.

Launch SGLang (also from a terminal):

```bash
python -m sglang.launch_server \
  --model-path Qwen/Qwen2.5-0.6B \
  --port 8000 \
  --mem-fraction-static 0.5
```

SGLang's parameter names differ slightly from vLLM:

| Parameter | vLLM equivalent | Purpose |
|:---|:---|:---|
| `--model-path` | `--model` | model path |
| `--port` | `--port` | server port |
| `--mem-fraction-static` | `--gpu-memory-utilization` | VRAM utilization fraction |

Once the server starts it listens on the same `/v1/chat/completions` endpoint. **The client code is identical to the vLLM call above** -- just change `VLLM_URL` in cell 13 to SGLang's port; the requests call, message structure, and streaming parameters all carry over. That is the great advantage of OpenAI-compatible protocols: switching inference backends requires no business-code changes, and you can even do "vLLM primary, SGLang fallback" failover on the business side.

### SGLang's structured output

SGLang goes deeper than vLLM on structured output. It has built-in constrained decoding, which can force the output to match a JSON schema or regex at generation time -- very useful when you need an LLM to emit parseable structured data.

Constrain the output with the `json_schema` parameter:

```python
response = requests.post(
    "http://localhost:8000/v1/chat/completions",
    json={
        "model": "Qwen/Qwen2.5-0.6B",
        "messages": [
            {"role": "user", "content": "Introduce the Python programming language as JSON."}
        ],
        "extra_body": {
            "json_schema": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "year": {"type": "integer"},
                    "paradigms": {"type": "array", "items": {"type": "string"}}
                },
                "required": ["name", "year", "paradigms"]
            }
        }
    }
)
```

During decode, SGLang dynamically masks tokens that would violate the schema, guaranteeing the output can be parsed by `json.loads`. This is far more reliable than asking the model to "try to output JSON" and patching it up afterward.

## 7. Choosing between vLLM and SGLang

Side-by-side comparison for tech selection:

| Dimension | vLLM | SGLang |
|:---|:---|:---|
| Core innovation | PagedAttention (paged KV Cache) | RadixAttention (prefix-sharing tree) |
| Prefix caching | Automatic Prefix Caching (added later) | RadixAttention (native) |
| Structured output | via outlines/xgrammar integration | native constrained decoding |
| Multi-LoRA switching | well supported | supported |
| Distributed inference | TP/PP | TP |
| Ecosystem maturity | larger community, more issues | fast-growing, paper-driven |
| Typical use case | high-concurrency general API | multi-turn dialogue, agents, structured output |

It is worth noting that the two frameworks are converging quickly -- vLLM has added prefix caching and structured output, and SGLang has optimized general throughput. Under newer versions, the selection gap is much smaller than a year ago. Which to pick ultimately depends on your workload: running a real benchmark beats any comparison table.

Three metrics matter when benchmarking: throughput (tokens/s), time-to-first-token (TTFT), and tail latency (P95/P99). Different workloads are sensitive to different metrics -- chatbots care about TTFT, offline batch only cares about throughput.

## 8. Deploying your own trained model: the core obstacle

Deploying Qwen2.5-0.6B above went smoothly because vLLM and SGLang ship a built-in implementation of `Qwen2ForCausalLM`. But if you take a model you trained yourself (the MiniGPT from Notebook 06, say) and try to deploy it, you will hit this error immediately:

```
ValueError: Model architectures ['MiniGPTForCausalLM'] are not supported
on the current platform. Supported architectures: ...
```

This error is the core obstacle: the inference framework does not recognize your custom architecture.

The `architectures` field in `config.json` is a string that tells the framework which Python class to instantiate. vLLM and SGLang ship common architectures (LlamaForCausalLM, Qwen2ForCausalLM, MistralForCausalLM, etc.) but cannot know in advance what your MiniGPT looks like -- how many layers, how attention is implemented, what the weight names are.

To make the framework recognize a new architecture, there are two paths:

- **Path A: register via HuggingFace transformers**. Wrap the model as a transformers-compatible `PreTrainedModel`, then register it with `AutoModelForCausalLM.register()`. Both vLLM and SGLang will auto-discover it through transformers integration
- **Path B: register directly inside vLLM**. Register a native vLLM model implementation in vLLM's model registry with the `@register_model` decorator. Better performance, but requires writing more vLLM-internal API code

The next two sections walk through each path, then we handle the custom vocabulary problem.

## 9. Path A: register via transformers

This is the recommended path: implement once, works with both vLLM and SGLang, and also lets you load the model directly with transformers and train it with Trainer.

Three things to do:

1. Write a `MiniGPTConfig` subclass of `PretrainedConfig` that records all architectural hyperparameters
2. Write a `MiniGPTForCausalLM` subclass of `PreTrainedModel` that implements forward
3. Register the classes with `AutoConfig.register` and `AutoModelForCausalLM.register`

Below we use the MiniGPT structure from Notebook 06 as the example, walking through the full registration flow. hidden_size and num_layers are kept small so it runs on CPU.

In [ ]:
# This cell runs on CPU
# pip install transformers torch

import torch
import torch.nn as nn
from transformers import PretrainedConfig, PreTrainedModel
from transformers import AutoConfig, AutoModelForCausalLM

# 1. Define the Config class: records all architectural hyperparameters
class MiniGPTConfig(PretrainedConfig):
    """MiniGPT configuration; records vocab size, layers, hidden_size, etc."""

    model_type = "minigpt"   # required; transformers uses this field to distinguish models

    def __init__(
        self,
        vocab_size=256,         # small vocab for CPU demo
        hidden_size=64,         # small hidden for CPU demo
        num_hidden_layers=2,    # two layers is enough for a demo
        num_attention_heads=4,
        intermediate_size=128,  # FFN intermediate dimension
        max_position_embeddings=128,
        rms_norm_eps=1e-6,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.intermediate_size = intermediate_size
        self.max_position_embeddings = max_position_embeddings
        self.rms_norm_eps = rms_norm_eps

print("MiniGPTConfig defined, model_type = 'minigpt'")
print("Defaults: hidden_size=64, num_layers=2, vocab_size=256")

In [ ]:
# 2. Define the Model class: implement forward; inherit PreTrainedModel for save/load

class RMSNorm(nn.Module):
    """RMSNorm matching LLaMA/Qwen."""
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        norm = x.pow(2).mean(-1, keepdim=True).add(self.eps).rsqrt()
        return x * norm * self.weight


class MiniGPTDecoderLayer(nn.Module):
    """Simplified decoder layer: Pre-norm + Self-Attention + residual, then Pre-norm + FFN + residual."""
    def __init__(self, config):
        super().__init__()
        self.norm1 = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.attn = nn.MultiheadAttention(
            config.hidden_size, config.num_attention_heads, batch_first=True
        )
        self.norm2 = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.ffn = nn.Sequential(
            nn.Linear(config.hidden_size, config.intermediate_size),
            nn.GELU(),
            nn.Linear(config.intermediate_size, config.hidden_size),
        )

    def forward(self, x, attn_mask=None):
        # Pre-norm + Attention + residual
        h = self.norm1(x)
        attn_out, _ = self.attn(h, h, h, attn_mask=attn_mask, need_weights=False)
        x = x + attn_out
        # Pre-norm + FFN + residual
        h = self.norm2(x)
        x = x + self.ffn(h)
        return x


class MiniGPTForCausalLM(PreTrainedModel):
    """Full MiniGPT language model. Inherits save_pretrained/load_pretrained from PreTrainedModel."""

    config_class = MiniGPTConfig   # tells transformers which Config class to use
    _no_split_modules = ["MiniGPTDecoderLayer"]

    def __init__(self, config):
        super().__init__(config)
        self.config = config

        # token embedding (no position embedding here; left as an exercise)
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        # stacked decoder layers
        self.layers = nn.ModuleList([
            MiniGPTDecoderLayer(config) for _ in range(config.num_hidden_layers)
        ])
        # final norm + lm_head
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)

        # PreTrainedModel requires: init weights then call post_init
        self.post_init()

    def forward(self, input_ids, attention_mask=None, **kwargs):
        # 1. token IDs -> embeddings
        x = self.embed_tokens(input_ids)
        # 2. build causal mask (lower triangle)
        T = input_ids.shape[1]
        causal_mask = torch.triu(
            torch.full((T, T), float("-inf"), device=x.device), diagonal=1
        )
        # 3. iterate over layers
        for layer in self.layers:
            x = layer(x, attn_mask=causal_mask)
        # 4. norm + lm_head -> logits
        x = self.norm(x)
        logits = self.lm_head(x)
        return {"logits": logits}


print("MiniGPTForCausalLM implemented")
print("Structure: embed -> 2 x decoder_layer -> norm -> lm_head")

In [ ]:
# 3. Register with transformers so AutoConfig and AutoModelForCausalLM recognize it

AutoConfig.register("minigpt", MiniGPTConfig)
AutoModelForCausalLM.register(MiniGPTConfig, MiniGPTForCausalLM)

print("Registered")
print("MiniGPT can now be loaded via AutoModelForCausalLM.from_pretrained")

# Verify: instantiate and count parameters
config = MiniGPTConfig()
model = MiniGPTForCausalLM(config)

n_params = sum(p.numel() for p in model.parameters())
print(f"\nParameter count: {n_params:,} ({n_params/1e6:.2f}M)")
print("Sample weight names:")
for name, _ in list(model.named_parameters())[:5]:
    print(f"  {name}")
print("  ...")

In [ ]:
# Save MiniGPT in HuggingFace format
# After saving, the directory can be loaded by vLLM just like Qwen2.5-0.6B

import os
import json
import tempfile

save_dir = tempfile.mkdtemp(prefix="minigpt-hf-")
model.save_pretrained(save_dir)

print("Saved directory contents:")
for f in sorted(os.listdir(save_dir)):
    size = os.path.getsize(os.path.join(save_dir, f))
    print(f"  {f:40s} {size/1024:.1f} KB")

# Inspect config.json -- note the architectures field
print("\nconfig.json contents:")
with open(os.path.join(save_dir, "config.json")) as fp:
    saved_config = json.load(fp)
print(json.dumps(saved_config, indent=2))

print(f"\n-> The architectures field was auto-filled as ['MiniGPTForCausalLM']")
print(f"-> vLLM reads this field when loading to decide which Python class to instantiate")

### Load the registered MiniGPT with vLLM

`config.json` now has `architectures = ["MiniGPTForCausalLM"]`. But vLLM does not know this name by default; we need transformers integration to make vLLM discover the class.

The simplest approach is to run the registration code (the `AutoConfig.register` and `AutoModelForCausalLM.register` from cell 23 above) in Python before vLLM starts. When vLLM loads the model, transformers' `AutoModel` will pick up the class.

Write a `register_model.py` and import it before launching vLLM:

```python
# register_model.py
from transformers import AutoConfig, AutoModelForCausalLM
from minigpt_model import MiniGPTConfig, MiniGPTForCausalLM  # the code we just wrote

AutoConfig.register("minigpt", MiniGPTConfig)
AutoModelForCausalLM.register(MiniGPTConfig, MiniGPTForCausalLM)
```

Then launch vLLM from a terminal:

```bash
# --trust-remote-code lets vLLM execute user code (the registration script)
vllm serve /tmp/minigpt-hf-xxxxx \
  --port 8000 \
  --enforce-eager
```

`--enforce-eager` disables CUDA Graph and runs in eager mode. A custom class like MiniGPT has not been adapted for CUDA Graph, and leaving it on would error; eager mode is slightly slower but more permissive.

If everything goes well, vLLM's log will show `Model architectures ['MiniGPTForCausalLM']` and the server is up. The client code is identical to the Qwen2.5-0.6B call earlier -- just change the model field to `MiniGPTForCausalLM`.

A few things to watch out for:

- Models registered through transformers use vLLM's "generic fallback" path, with worse performance than native optimization (PagedAttention may not apply)
- If the model uses non-standard attention (MLA, Sliding Window, etc.), vLLM needs extra hints on how to handle it
- For maximum performance, take Path B

## 10. Path B: register directly inside vLLM

Path A's limitation is performance -- transformers-wrapped models go through vLLM's generic execution path and miss out on PagedAttention, FlashAttention, and other GPU optimizations. To deploy a custom model in a high-concurrency production environment, you need to write a native vLLM implementation.

This path requires more vLLM-internal API: subclass `nn.Module`, implement `forward`, implement `load_weights` (map checkpoint weight names to model parameters), and register with `@register_model`. More code, but you get every vLLM optimization.

Below is a skeleton showing the structure of a native vLLM model implementation. For a full example, see `vllm/model_executor/models/llama.py` in the vLLM source.

Below is a skeleton showing the structure of a native vLLM model implementation. For a full example, see `vllm/model_executor/models/llama.py` in the vLLM source.

```python
# Skeleton: requires vllm and a GPU to actually run
# from vllm.model_executor.models.registry import register_model
# from vllm.model_executor.layers.sampler import Sampler
# from vllm.model_executor.layers.embedding import VocabParallelEmbedding
# from vllm.model_executor.layers.linear import ParallelLMHead
# from vllm.attention import Attention

@register_model("MiniGPTForCausalLM")
class MiniGPTForCausalLM_vLLM(nn.Module):
    # Native vLLM implementation: use vLLM's Attention layer and Sampler directly.
    # Key differences:
    # - Cannot use nn.MultiheadAttention; use vllm.attention.Attention instead
    # - forward signature changes: receives positions and kv_caches (managed by vLLM)
    # - Must implement load_weights for checkpoint name mapping

    def __init__(self, config, *, cache_config=None, quant_config=None):
        super().__init__()
        self.config = config
        # 1. embedding (vLLM's parallel version, supports TP sharding)
        self.embed_tokens = VocabParallelEmbedding(
            config.vocab_size, config.hidden_size
        )
        # 2. decoder layers (using vLLM's Attention)
        self.layers = nn.ModuleList([
            MiniGPTDecoderLayer_vLLM(config, cache_config, quant_config)
            for _ in range(config.num_hidden_layers)
        ])
        # 3. final norm + lm_head
        self.norm = RMSNorm(config.hidden_size, config.rms_norm_eps)
        self.lm_head = ParallelLMHead(config.vocab_size, config.hidden_size)
        # 4. sampler (built into vLLM, handles batching automatically)
        self.sampler = Sampler()

    def forward(self, input_ids, positions, kv_caches, **kwargs):
        # vLLM forward signature: positions is the absolute position of each token,
        # kv_caches is the per-layer KV Cache (managed by vLLM's PagedAttention).
        x = self.embed_tokens(input_ids)
        for i, layer in enumerate(self.layers):
            x = layer(x, positions, kv_caches[i])
        x = self.norm(x)
        logits = self.lm_head(x)
        return logits

    def load_weights(self, weights):
        # Load weights from a checkpoint.
        # weights is an iterator yielding (name, tensor) pairs.
        # Here you map checkpoint names to the model's nn.Module parameters.
        params_dict = dict(self.named_parameters())
        for name, weight in weights:
            if name in params_dict:
                param = params_dict[name]
                weight_loader = getattr(param, "weight_loader", default_weight_loader)
                weight_loader(param, weight)
```

Core changes versus Path A:

- `nn.MultiheadAttention` -> `vllm.attention.Attention`
- forward signature changes: `positions` and `kv_caches` are managed by vLLM
- must implement `load_weights` for checkpoint name mapping

After this, vLLM can use PagedAttention, FlashAttention, and all other optimizations. The cost is code volume -- each new model takes about 200-400 lines.

## 11. Custom vocabulary

Beyond a new architecture, custom models usually come with a new vocabulary. If you trained with the hand-written BPE Tokenizer from Notebook 02, vLLM and SGLang will not recognize it -- they only understand the `tokenizer.json` format from the HuggingFace `tokenizers` library.

To make a custom tokenizer loadable by inference frameworks, you need to convert it to HF format. The cleanest approach is to train with the HF `tokenizers` library from the start and get a `tokenizer.json` directly; if you already have a custom format, you need to convert manually.

HF `tokenizers` BPE training API:

```python
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

tokenizer = Tokenizer(BPE(unk_token="<unk>"))
tokenizer.pre_tokenizer = Whitespace()
trainer = BpeTrainer(
    vocab_size=8000,
    special_tokens=["<unk>", "<bos>", "<eos>", "<pad>"]
)
tokenizer.train(files=["corpus.txt"], trainer=trainer)
tokenizer.save("tokenizer.json")
```

The saved `tokenizer.json` can be dropped into the model directory; vLLM and SGLang will use it automatically when loading the model.

If you already have a non-HF tokenizer (the hand-written BPE from Notebook 02, say), the conversion approach is to read out the merge rules and vocab, then rewrite them in HF format. HF BPE format has strict field requirements (`model.vocab`, `model.merges`, `pre_tokenizer`, `decoder`, etc.); vLLM will error if any are missing.

## 12. End-to-end: deploy MiniGPT

Putting the previous steps together, the full deployment pipeline:

```text
1. Define MiniGPTConfig + MiniGPTForCausalLM (subclass PreTrainedModel)
2. Register with AutoConfig.register / AutoModelForCausalLM.register
3. Train or load weights
4. model.save_pretrained("minigpt-weights/")
5. tokenizer.save("minigpt-weights/tokenizer.json")
6. Write register_model.py and run the registration once
7. vllm serve minigpt-weights/ --trust-remote-code --enforce-eager
8. Send requests with an OpenAI-compatible client
```

If step 7 starts the server cleanly, step 8 calls MiniGPT just like Qwen2.5-0.6B -- the client code is unchanged.

The final directory looks like:

```text
minigpt-weights/
├── config.json               # architecture config, architectures=["MiniGPTForCausalLM"]
├── model.safetensors         # weights
├── tokenizer.json            # tokenizer
├── tokenizer_config.json     # tokenizer behavior (special tokens, etc.)
└── generation_config.json    # default generation params
```

This directory can be packaged and shared; others can `vllm serve minigpt-weights/` -- as long as they also register the MiniGPT model class. That is why open-source models ship not just weights but also the Python code for the model class (e.g., a `modeling_xxx.py` file in HuggingFace model repos).

## Summary

### Inference framework basics (Sections 1-2)

- [ ] HuggingFace transformers' static batching has low throughput under variable-length requests
- [ ] Continuous batching lets new requests join or leave the batch at every decode step
- [ ] PagedAttention manages KV Cache as pages, reducing memory fragmentation (proposed by vLLM)
- [ ] RadixAttention reuses KV Cache across shared prefixes (proposed by SGLang)

### Deploying an off-the-shelf model (Sections 3-7)

- [ ] Two vLLM usage modes: `LLM` offline inference and `vllm serve` HTTP service
- [ ] vLLM's API is fully compatible with OpenAI Chat Completions
- [ ] SGLang's workflow is almost identical to vLLM; client code is reusable without changes
- [ ] Key parameters: `gpu_memory_utilization`, `max_model_len`, `dtype`
- [ ] Streaming output pushes tokens via the SSE protocol
- [ ] Selection: vLLM for raw throughput, SGLang for structured output and shared prefixes

### Deploying your own model (Sections 8-12)

- [ ] Core obstacle: the framework does not recognize the new architecture (`architectures` field in `config.json`)
- [ ] Path A: register via transformers (simple, decent performance, recommended for getting started)
- [ ] Path B: register natively in vLLM with `@register_model` (complex, best performance)
- [ ] Custom vocabularies must be converted to HF `tokenizers` `tokenizer.json` format
- [ ] End-to-end flow: define classes -> register -> train -> save_pretrained -> vllm serve

### One-line takeaway

Deploying an off-the-shelf model is an engineering problem; deploying your own model is a "how do you make the inference framework recognize your code" problem -- the difficulty of the former is tuning and load testing, the difficulty of the latter is adapting to the transformers/vLLM interface conventions.

### References

- vLLM docs: https://docs.vllm.ai/
- vLLM GitHub: https://github.com/vllm-project/vllm
- Adding a new model to vLLM: https://docs.vllm.ai/en/latest/models/adding_model.html
- SGLang docs: https://docs.sglang.ai/
- SGLang GitHub: https://github.com/sgl-project/sglang
- PagedAttention paper (SOSP 2023): https://arxiv.org/abs/2309.06180
- SGLang paper: https://arxiv.org/abs/2312.07104
- Orca paper (origin of continuous batching, OSDI 2022): https://www.usenix.org/conference/osdi22/presentation/yu
- HuggingFace tokenizers docs: https://huggingface.co/docs/tokenizers
- HuggingFace custom models guide: https://huggingface.co/docs/transformers/custom_models
- Qwen2.5 model card: https://huggingface.co/Qwen/Qwen2.5-0.6B

## Homework

> You can ask AI for explanations, sub-steps, and direction checks, but it is not recommended to have AI "do the whole problem for you."

### Homework 1: The throughput advantage of continuous batching

Read the static/continuous batching simulation in cell 3. When `batch_size` grows to 16 and the number of requests is still 8, does the total time for static batching grow, shrink, or stay the same? Why?

Hint: look at the line `batch_time = max(batch)` in `simulate_static_batching` -- what determines a batch's time?

```python
# Homework 1: fill in True or False
answer_1 = None  # True means "will change", False means "stays the same"

# Your reasoning:
reason_1 = ""

assert answer_1 is not None, "Please fill in True or False"
assert isinstance(answer_1, bool), "Please fill in True or False"
if answer_1 is False:
    print("✅ Homework 1 passed")
    print("   batch_size=16 but only 8 requests -> only one batch")
    print("   batch_time = max(all 8 request lengths)")
    print("   Compare with batch_size=4 split into two batches (max(first 4) + max(last 4))")
    print("   The key point: when batch_size exceeds request count, extra slots add no speedup")
else:
    print("Hint: when batch_size exceeds the request count, how does the number of batches change?")
```

### Homework 2: How key vLLM parameters affect memory

The `LLM` class has three parameters that affect memory: `gpu_memory_utilization`, `max_model_len`, `dtype`. Assume the GPU has 24 GB total VRAM and the model weights take 4 GB. Which configuration gives vLLM the largest KV Cache pool?

```python
# Homework 2: pick A / B / C / D
answer_2 = ""

# A. gpu_memory_utilization=0.5, max_model_len=2048, dtype="float16"
# B. gpu_memory_utilization=0.9, max_model_len=8192, dtype="float16"
# C. gpu_memory_utilization=0.9, max_model_len=2048, dtype="float16"
# D. gpu_memory_utilization=0.9, max_model_len=2048, dtype="float32"

assert not answer_2.startswith("HERE"), "Pick A/B/C/D"
assert answer_2 in "ABCD", "Pick one of A/B/C/D"

if answer_2 == "C":
    print("✅ Homework 2 passed")
    print("   KV Cache pool ~ gpu_memory_utilization x total VRAM - weight size")
    print("   C: 0.9 x 24 - 4 = 17.6 GB, the largest")
    print("   D uses float32, doubling weights to 8 GB, which actually shrinks the pool")
    print("   B's max_model_len is large but does not change the pool (only the per-request ceiling)")
else:
    print(f"You picked {answer_2}")
    print("Hint: max_model_len sets per-request ceiling, not pool size")
```

### Homework 3: Registration order for a custom architecture

To load a brand-new architecture `MyAwesomeLLM` with vLLM, in what order should the following steps run?

```python
# Homework 3: fill the letters in the correct order
answer_3 = ""

# A. vllm serve my-model/
# B. Implement MyAwesomeLLMForCausalLM(PreTrainedModel) class
# C. model.save_pretrained("my-model/")
# D. AutoModelForCausalLM.register(MyAwesomeLLMConfig, MyAwesomeLLMForCausalLM)
# E. Implement MyAwesomeLLMConfig(PretrainedConfig) class
# F. Train or load weights

assert len(answer_3) == 6, "Please fill 6 letters in the correct order"
assert set(answer_3) == set("ABCDEF"), "Letters cannot repeat or be missing"

if answer_3 == "EBFDCA":
    print("✅ Homework 3 passed")
    print("   Order: E (Config) -> B (Model) -> F (weights) -> D (register) -> C (save) -> A (launch)")
    print("   Key: Config must exist before Model can be defined; after registration it can save/load correctly")
else:
    print(f"Your order: {answer_3}")
    print("Hint: define Config first, then Model; register before save and launch")
```